In [1]:
# ==========================
# CELDA 1: CONFIGURACIÓN Y LIBRERÍAS
# ==========================

import sys
from pathlib import Path
import polars as pl
import gc

# Ruta del proyecto
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Librería para manejo de Parquet
from scr.python.clase_datalake import ParquetStore

# Recarga automática de módulos (útil durante desarrollo)
%load_ext autoreload
%autoreload 2

# --------------------------
# VARIABLES DE ENTRADA / RUTAS
# --------------------------
datalake_dir = "../../data/rawparquet"   # Carpeta con los Parquets originales
dataoutput = "../../data/interim"        # Carpeta para guardar parquet final

# Rangos de años y meses
anio_ini = 1995
anio_def = 2023
anio_fin = 2025
meses_list = None  # None = todos los meses

# Ámbitos
# Mapeo de ámbitos a códigos de comunidad
'''mapeo_ambito_cod_comunidad = {
    "madrid": 15,
    "espana": 99
}'''

mapeo_ambito_cod_comunidad = {
    "nodeterminado": 0,
    "andalucia": 1,
    "aragon": 2,
    "asturias": 3,
    "baleares": 4,
    "canarias": 5,
    "cantabria": 17,
    "castillalamancha": 7,
    "castillayleon": 6,
    "cataluna": 8,
    "galicia": 10,
    "madrid": 15,
    "murcia": 11,
    "navarra": 12,
    "paisvasco": 14,
    "rioja": 16,
    "valencia": 13,
    "ceuta": 51,
    "melilla": 52,
    "espana": 99
}

ambitos = list(mapeo_ambito_cod_comunidad.keys())

# Dominios a procesar
dominios = [
    "euros_taric",
    "euros_taric_pais",
    "euros_pais",
    "kg_taric",
    "kg_taric_pais",
    "kg_pais",
    "euros_sectores",
    "euros_sectores_pais"
]

# Columnas extra por dominio (para alinear esquemas)
extra_cols = {
    "euros_taric": {"pais": 0},
    "euros_taric_pais": {},  
    "euros_pais": {"cod_taric": 0, "nivel_taric": 0, "cod_sector_economico":"0", "nivel_sector_economico":0},
    "kg_taric": {"pais": 0},
    "kg_taric_pais": {},
    "kg_pais": {"cod_taric": 0, "nivel_taric": 0},
    "euros_sectores": {"pais": 0},
    "euros_sectores_pais": {},
    "euros_pais_bis": {}
}

# --------------------------
# FUNCIONES AUXILIARES
# --------------------------
def align_schema(df: pl.LazyFrame | pl.DataFrame, schema: dict[str, pl.DataType]) -> pl.LazyFrame:
    """
    Alinear un DataFrame o LazyFrame al esquema definido:
    - Añade columnas faltantes
    - Fuerza los tipos
    - Reordena columnas
    """
    # Añadir columnas faltantes
    for col, dtype in schema.items():
        if col not in df.columns:
            df = df.with_columns(pl.lit(None).cast(dtype).alias(col))
    
    # Forzar tipos
    df = df.with_columns([pl.col(col).cast(dtype) for col, dtype in schema.items()])

    # Reordenar
    return df.select(list(schema.keys()))

# --------------------------
# SCHEMAS DE DOMINIOS
# --------------------------
TARIC_EUROS_SCHEMA = {
    "flujo": pl.Int32,
    "anio": pl.Int32,
    "mes": pl.Int32,
    "estado": pl.Int32,
    "pais": pl.Int32,
    "nivel_taric": pl.Int32,
    "cod_taric": pl.Int64,
    "euros": pl.Float64,
}

SECTORES_EUROS_SCHEMA = {
    "flujo": pl.Int32,
    "anio": pl.Int32,
    "mes": pl.Int32,
    "estado": pl.Int32,
    "pais": pl.Int32,
    "nivel_sector_economico": pl.Int32,
    "cod_sector_economico": pl.Utf8,
    "euros": pl.Float64,
}

TARIC_KG_SCHEMA = {
    "flujo": pl.Int32,
    "anio": pl.Int32,
    "mes": pl.Int32,
    "estado": pl.Int32,
    "pais": pl.Int32,
    "nivel_taric": pl.Int32,
    "cod_taric": pl.Int64,   
    "kilogramos": pl.Float64,
}

In [2]:
# ==========================
# CELDA 2: UNIR PARQUETS EN UN LAZYFRAME Y GUARDAR
# ==========================

final_lfs = {}

with ParquetStore(datalake_dir) as store:

    # --------------------------
    # Cargar totales CCAA
    # --------------------------
    lf_totales = store.merge_parquet_range(
        ambito="",
        dominio="totalesccaa",
        anio_inicio=anio_ini,
        anio_definitivo=anio_def,
        anio_fin=anio_fin,
        meses=meses_list,
        lazy=True
    )
    
    # Guardar totales CCAA como un LazyFrame más
    final_lfs["totalesccaa"] = lf_totales

    # --------------------------
    # Procesar dominios por ambito
    # --------------------------
    for ambito in ambitos:
        print(f"\n📍 Procesando ámbito: {ambito}")
        
        # Código de comunidad correspondiente
        cod_comunidad = mapeo_ambito_cod_comunidad[ambito]

        # ---------------- TARIC - EUROS ----------------
        dfs_list = []
        for dominio in ["euros_taric", "euros_taric_pais", "euros_pais"]:
            lf = store.merge_parquet_range(
                ambito=ambito,
                dominio=dominio,
                anio_inicio=anio_ini,
                anio_definitivo=anio_def,
                anio_fin=anio_fin,
                meses=meses_list,
                lazy=True
            )
            if dominio in extra_cols and extra_cols[dominio]:
                for col_name, value in extra_cols[dominio].items():
                    lf = lf.with_columns(pl.lit(value).alias(col_name))
            lf = align_schema(lf, TARIC_EUROS_SCHEMA)
            dfs_list.append(lf)

        # Añadir totales de CCAA
        total_taric_euros = (
            lf_totales
            .filter(pl.col("cod_comunidad") == cod_comunidad)
            .with_columns(
                pl.lit(0).cast(pl.Int64).alias("cod_taric"),
                pl.lit(0).cast(pl.Int32).alias("nivel_taric"),
                pl.lit(0).cast(pl.Int32).alias("pais")
            )
            .drop("cod_comunidad")
        )
        total_taric_euros = align_schema(total_taric_euros, TARIC_EUROS_SCHEMA)
        dfs_list.append(total_taric_euros)

        final_lfs[f"{ambito}_euros_taric"] = pl.concat(dfs_list, how="vertical")
        del dfs_list, total_taric_euros
        gc.collect()

        # ---------------- TARIC - KG ----------------
        dfs_list = []
        for dominio in ["kg_taric", "kg_taric_pais", "kg_pais"]:
            lf = store.merge_parquet_range(
                ambito=ambito,
                dominio=dominio,
                anio_inicio=anio_ini,
                anio_definitivo=anio_def,
                anio_fin=anio_fin,
                meses=meses_list,
                lazy=True
            )
            if dominio in extra_cols and extra_cols[dominio]:
                for col_name, value in extra_cols[dominio].items():
                    lf = lf.with_columns(pl.lit(value).alias(col_name))
            lf = align_schema(lf, TARIC_KG_SCHEMA)
            dfs_list.append(lf)

        total_taric_kg = (
            lf_totales
            .filter(pl.col("cod_comunidad") == cod_comunidad)
            .with_columns(
                pl.lit(0).cast(pl.Int64).alias("cod_taric"),
                pl.lit(0).cast(pl.Int32).alias("nivel_taric"),
                pl.lit(0).cast(pl.Int32).alias("pais")
            )
            .drop("cod_comunidad")
        )
        total_taric_kg = align_schema(total_taric_kg, TARIC_KG_SCHEMA)
        dfs_list.append(total_taric_kg)

        final_lfs[f"{ambito}_kg_taric"] = pl.concat(dfs_list, how="vertical")
        del dfs_list, total_taric_kg
        gc.collect()

        # ---------------- SECTORES - EUROS ----------------
        dfs_list = []
        for dominio in ["euros_sectores", "euros_sectores_pais", "euros_pais"]:
            lf = store.merge_parquet_range(
                ambito=ambito,
                dominio=dominio,
                anio_inicio=anio_ini,
                anio_definitivo=anio_def,
                anio_fin=anio_fin,
                meses=meses_list,
                lazy=True
            )
            if dominio in extra_cols and extra_cols[dominio]:
                for col_name, value in extra_cols[dominio].items():
                    lf = lf.with_columns(pl.lit(value).alias(col_name))
            lf = align_schema(lf, SECTORES_EUROS_SCHEMA)
            dfs_list.append(lf)

        total_sectores = (
            lf_totales
            .filter(pl.col("cod_comunidad") == cod_comunidad)
            .with_columns(
                pl.lit("0").alias("cod_sector_economico"),
                pl.lit(0).cast(pl.Int32).alias("nivel_sector_economico"),
                pl.lit(0).cast(pl.Int32).alias("pais")
            )
            .drop("cod_comunidad")
        )
        total_sectores = align_schema(total_sectores, SECTORES_EUROS_SCHEMA)
        dfs_list.append(total_sectores)

        final_lfs[f"{ambito}_euros_sectores"] = pl.concat(dfs_list, how="vertical")
        del dfs_list, total_sectores
        gc.collect()

print(f"\n✅ Procesados {len(final_lfs)} LazyFrames")

# --------------------------
# Guardar LazyFrames a parquet/csv
# --------------------------
print("\n💾 Guardando archivos...")
for name, lf in final_lfs.items():
    # Caso especial para totales CCAA (guardar como CSV)
    if name == "totalesccaa":
        # Crear carpeta para totales CCAA
        totales_dir = Path(dataoutput) / "totalesccaa"
        totales_dir.mkdir(parents=True, exist_ok=True)
        path = totales_dir / "totalesccaa.csv"
        
        # Materializar LazyFrame
        df = lf.collect()
        
        # Renombrar columna 'anio' a 'año' si existe
        if 'anio' in df.columns:
            df = df.rename({'anio': 'año'})
        
        # Escribir CSV
        df.write_csv(path)
        print(f"  ✅ {name}.csv")
        
    else:
        # Extraer el nombre del ámbito del nombre del archivo
        ambito_name = name.split('_')[0]
        
        # Crear carpeta del ámbito si no existe
        ambito_dir = Path(dataoutput) / ambito_name
        ambito_dir.mkdir(parents=True, exist_ok=True)
        
        # Construir path dentro de la carpeta del ámbito
        path = ambito_dir / f"{name}.parquet"

        # Materializar LazyFrame
        df = lf.collect()

        # Renombrar columna 'anio' a 'año' si existe
        if 'anio' in df.columns:
            df = df.rename({'anio': 'año'})

        # Escribir parquet
        df.write_parquet(path)
        print(f"  ✅ {name}.parquet")
    
    del df
    gc.collect()

print(f"\n🎉 Proceso completado. Archivos guardados en {dataoutput}")

2026-02-02 19:23:23 - INFO - Agregando nacional/totalesccaa 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:23 - INFO - Agregando nodeterminado/euros_taric 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
/tmp/ipykernel_116354/2810863872.py:102: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if col not in df.columns:
2026-02-02 19:23:23 - INFO - Agregando nodeterminado/euros_taric_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:23 - INFO - Agregando nodeterminado/euros_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:23 - INFO - Agregando nodeterminado/kg_taric 


📍 Procesando ámbito: nodeterminado


2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:23 - INFO - Agregando nodeterminado/kg_taric_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:23 - INFO - Agregando nodeterminado/kg_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:23 - INFO - Agregando nodeterminado/euros_sectores 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:23 - INFO - Agregando nodeterminado/euros_sectores_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:23 - INFO - Agregando nodeterminado/euros_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:23 - INFO - Agregando andalucia/euros_taric 1995-2025 (def hasta 2023)
2026-02-02 19:23:23 - INFO - ✓


📍 Procesando ámbito: andalucia


2026-02-02 19:23:24 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:24 - INFO - Agregando andalucia/kg_taric_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:24 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:24 - INFO - Agregando andalucia/kg_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:24 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:24 - INFO - Agregando andalucia/euros_sectores 1995-2025 (def hasta 2023)
2026-02-02 19:23:24 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:24 - INFO - Agregando andalucia/euros_sectores_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:24 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:24 - INFO - Agregando andalucia/euros_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:24 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:24 - INFO - Agregando aragon/euros_taric 1995-2025 (def hasta 2023)
2026-02-02 19:23:24 - INFO - ✓ 371 parquets añadidos 

...


📍 Procesando ámbito: melilla


2026-02-02 19:23:32 - INFO - Agregando melilla/euros_sectores 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando melilla/euros_sectores_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando melilla/euros_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando espana/euros_taric 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando espana/euros_taric_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando espana/euros_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación



📍 Procesando ámbito: espana


2026-02-02 19:23:32 - INFO - Agregando espana/kg_taric 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando espana/kg_taric_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando espana/kg_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando espana/euros_sectores 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando espana/euros_sectores_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación
2026-02-02 19:23:32 - INFO - Agregando espana/euros_pais 1995-2025 (def hasta 2023)
2026-02-02 19:23:32 - INFO - ✓ 371 parquets añadidos a la agregación



✅ Procesados 61 LazyFrames

💾 Guardando archivos...
  ✅ totalesccaa.csv
  ✅ nodeterminado_euros_taric.parquet
  ✅ nodeterminado_kg_taric.parquet
  ✅ nodeterminado_euros_sectores.parquet
  ✅ andalucia_euros_taric.parquet
  ✅ andalucia_kg_taric.parquet
  ✅ andalucia_euros_sectores.parquet
  ✅ aragon_euros_taric.parquet
  ✅ aragon_kg_taric.parquet
  ✅ aragon_euros_sectores.parquet
  ✅ asturias_euros_taric.parquet
  ✅ asturias_kg_taric.parquet
  ✅ asturias_euros_sectores.parquet
  ✅ baleares_euros_taric.parquet
  ✅ baleares_kg_taric.parquet
  ✅ baleares_euros_sectores.parquet
  ✅ canarias_euros_taric.parquet
  ✅ canarias_kg_taric.parquet
  ✅ canarias_euros_sectores.parquet
  ✅ cantabria_euros_taric.parquet
  ✅ cantabria_kg_taric.parquet
  ✅ cantabria_euros_sectores.parquet
  ✅ castillalamancha_euros_taric.parquet
  ✅ castillalamancha_kg_taric.parquet
  ✅ castillalamancha_euros_sectores.parquet
  ✅ castillayleon_euros_taric.parquet
  ✅ castillayleon_kg_taric.parquet
  ✅ castillayleon_euros